# <font color="#418FDE" size="6.5" uppercase>**Signalmodelle**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Erzeugen Fenster und Statistik- sowie Frequenzmerkmale aus kleinen Signalen. 
- Trainieren scikit-learn-Pipelines für Signal-Klassifikation oder Regression. 
- Bewerten Signalmodelle zeitlich korrekt und vergleichen sie mit Baselines. 


## **1. Signalmerkmale aus Fenstern**

### **1.1. Fenster mit Labels**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_B/image_01_01.jpg?v=1784805288" width="250">



>* Signale in kurze, lernbare Fenster teilen
>* Labels machen Fenster zu Trainingsbeispielen

>* Fensterlänge steuert Kontext und Labelschärfe
>* Überlappung erhöht Beispiele, erschwert Bewertung

>* Labels bei Übergängen eindeutig festlegen
>* Regeln konsistent und echtzeitgerecht anwenden



In [ ]:
#@title Python-Code - Fenster mit Labels

# Dieses Beispiel erzeugt gelabelte Signalfenster.
# Fenster machen lange Signale lernbar.
# Die Ausgabe zeigt Fensterlabels und Merkmale.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Wir erzeugen ein kleines künstliches Sensorsignal.
sample_rate_hz = 10
seconds = 12
sample_count = sample_rate_hz * seconds

# Die Zeitachse nutzt Sekunden als verständliche Einheit.
time_seconds = np.arange(sample_count) / sample_rate_hz
signal = np.sin(2 * np.pi * 0.8 * time_seconds)

# Ab Sekunde sechs wird die Bewegung stärker.
labels_per_sample = np.where(time_seconds < 6, "ruhig", "aktiv")
signal = signal + np.where(labels_per_sample == "aktiv", 0.8, 0.0)

# Überlappende Fenster liefern mehrere Trainingsbeispiele.
window_size = 20
step_size = 10
starts = np.arange(0, sample_count - window_size + 1, step_size)

# Diese Prüfung schützt vor unpassenden Fenstergrößen.
if len(starts) == 0:
    raise ValueError("Die Fenstergröße ist größer als das Signal.")

# Jedes Fenster bekommt das häufigste Label seiner Messpunkte.
rows = []
for start in starts:
    end = start + window_size
    window_signal = signal[start:end]
    window_labels = labels_per_sample[start:end]
    label_values, label_counts = np.unique(window_labels, return_counts=True)
    majority_label = label_values[np.argmax(label_counts)]
    rows.append(
        {
            "start_s": round(float(time_seconds[start]), 1),
            "label": majority_label,
            "mean": round(float(np.mean(window_signal)), 2),
            "std": round(float(np.std(window_signal)), 2),
        }
    )

# Die Tabelle zeigt aus Signalabschnitten entstandene Beispiele.
features = pd.DataFrame(rows)
print("Erste gelabelte Fenster mit einfachen Merkmalen:")
print(features.head(5).to_string(index=False))

# Die Grafik markiert die Fensterstarts im Signal.
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(time_seconds, signal, label="Signal")
ax.scatter(features["start_s"], signal[starts], color="red", label="Fensterstart")
ax.set_title("Signal wird in überlappende Fenster zerlegt")
ax.set_xlabel("Zeit in Sekunden")
ax.set_ylabel("Sensorwert")
ax.legend()
plt.show()



### **1.2. Statistische Kennwerte**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_B/image_01_02.jpg?v=1784805286" width="250">



>* Kennwerte fassen Signalfenster kompakt zusammen
>* Sie machen lokale Muster analysierbar

>* Lage, Streuung und Form beschreiben Fenster
>* Robuste Kennwerte mindern Ausreißer-Einfluss

>* Fensterlänge passend zu Signal und Aufgabe wählen
>* Einheitliche Kennwerte liefern vergleichbare Merkmalsvektoren



In [ ]:
#@title Python-Code - Statistische Kennwerte

# Dieses Beispiel berechnet Kennwerte aus Signalfenstern.
# Fenster verdichten Rohwerte zu vergleichbaren Merkmalen.
# Die Grafik zeigt Signal und Fenstermerkmale.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ein kleines künstliches Sensorsignal bleibt vollständig reproduzierbar.
rng = np.random.default_rng(42)
sample_index = np.arange(60)
base_signal = np.sin(2 * np.pi * sample_index / 20)
noise = rng.normal(0, 0.15, size=sample_index.size)

# Ein kurzer Ausschlag macht Extremwerte und Spannweite sichtbar.
signal = base_signal + noise
signal[32:36] = signal[32:36] + 1.4
window_size = 12

# Die Fenster werden ohne Überlappung aus dem Signal geschnitten.
usable_length = signal.size - signal.size % window_size
windows = signal[:usable_length].reshape(-1, window_size)
window_ids = np.arange(windows.shape[0])

# Diese Prüfung schützt vor unpassenden Fenstergrößen.
if windows.shape[1] != window_size:
    raise ValueError("Die Fenstergröße passt nicht zum Signal.")

# Pro Fenster entstehen wenige gut interpretierbare Kennwerte.
features = pd.DataFrame({
    "Mittelwert": windows.mean(axis=1),
    "Std": windows.std(axis=1),
    "Minimum": windows.min(axis=1),
    "Maximum": windows.max(axis=1),
    "Spannweite": windows.max(axis=1) - windows.min(axis=1),
})

# Gerundete Werte sind für den ersten Überblick leichter lesbar.
rounded_features = features.round(2)
rounded_features.index.name = "Fenster"
print("Statistische Kennwerte pro Fenster:")
print(rounded_features.to_string())

# Die Spannweite wird jedem Fensterzentrum zugeordnet.
window_centers = window_ids * window_size + window_size / 2 - 0.5
range_values = features["Spannweite"].to_numpy()

# Eine einzige Grafik verbindet Rohsignal und berechnete Merkmale.
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sample_index, signal, marker="o", label="Signalwert")
ax.plot(window_centers, range_values, marker="s", label="Spannweite je Fenster")

ax.set_title("Statistische Kennwerte aus kurzen Signalfenstern")
ax.set_xlabel("Messpunkt")
ax.set_ylabel("Signalwert oder Spannweite")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()



### **1.3. Frequenzen im Fenster**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_B/image_01_03.jpg?v=1784805290" width="250">



>* Frequenzen zeigen zeitliche Muster im Fenster
>* Sie helfen, Aktivitäten und Ereignisse zu unterscheiden

>* Dominante Frequenzen zeigen wichtige Signalrhythmen.
>* Fensterlänge bestimmt Zeit- und Frequenzgenauigkeit.

>* Rauschen, Ränder und Abtastrate beeinflussen Frequenzen
>* Frequenzmerkmale zeigen passende zeitliche Rhythmen



In [ ]:
#@title Python-Code - Frequenzen im Fenster

# Dieses Beispiel zeigt Frequenzen in kurzen Signalfenstern.
# Eine FFT macht dominante Schwingungen sichtbar.
# Die Ausgabe vergleicht zwei Fenstermerkmale.

import numpy as np
import matplotlib.pyplot as plt

# Wir erzeugen ein kleines Signal mit zwei Abschnitten.
sampling_rate_hz = 50
window_seconds = 2
samples_per_window = sampling_rate_hz * window_seconds

# Die Zeitachse beschreibt Messzeitpunkte in Sekunden.
time = np.arange(2 * samples_per_window) / sampling_rate_hz
first_part = np.sin(2 * np.pi * 2 * time[:samples_per_window])
second_part = np.sin(2 * np.pi * 8 * time[:samples_per_window])

# Beide Abschnitte werden zu einem Signal verbunden.
signal = np.concatenate([first_part, second_part])
if signal.size != 2 * samples_per_window:
    raise ValueError("Das Signal hat nicht die erwartete Länge.")

# Diese Funktion berechnet Frequenzmerkmale für ein Fenster.
def frequency_features(window, sampling_rate_hz):
    window = np.asarray(window)
    window = window - np.mean(window)
    spectrum = np.fft.rfft(window)

    magnitudes = np.abs(spectrum)
    frequencies = np.fft.rfftfreq(window.size, d=1 / sampling_rate_hz)
    magnitudes[0] = 0

    dominant_index = np.argmax(magnitudes)
    dominant_frequency = frequencies[dominant_index]
    total_energy = np.sum(magnitudes ** 2)

    high_mask = frequencies >= 5
    high_energy_share = np.sum(magnitudes[high_mask] ** 2) / total_energy
    return dominant_frequency, high_energy_share

# Wir schneiden das Signal in zwei gleich lange Fenster.
window_a = signal[:samples_per_window]
window_b = signal[samples_per_window:]
features_a = frequency_features(window_a, sampling_rate_hz)
features_b = frequency_features(window_b, sampling_rate_hz)

# Die gedruckten Merkmale zeigen den Unterschied der Fenster.
print("Fenster A: dominante Frequenz =", round(features_a[0], 1), "Hz")
print("Fenster A: Anteil hoher Energie =", round(features_a[1], 2))
print("Fenster B: dominante Frequenz =", round(features_b[0], 1), "Hz")
print("Fenster B: Anteil hoher Energie =", round(features_b[1], 2))

# Für die Grafik berechnen wir das Spektrum beider Fenster.
frequencies = np.fft.rfftfreq(samples_per_window, d=1 / sampling_rate_hz)
spectrum_a = np.abs(np.fft.rfft(window_a - np.mean(window_a)))
spectrum_b = np.abs(np.fft.rfft(window_b - np.mean(window_b)))

# Die Grafik zeigt, wo die stärksten Frequenzen liegen.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(frequencies, spectrum_a, label="Fenster A: 2 Hz")
ax.plot(frequencies, spectrum_b, label="Fenster B: 8 Hz")

# Achsen und Legende machen die Frequenzmerkmale lesbar.
ax.set_title("Frequenzanteile in zwei Signalfenstern")
ax.set_xlabel("Frequenz in Hz")
ax.set_ylabel("Stärke im Spektrum")
ax.legend()
plt.show()



## **2. Zeitlich korrekt trainieren**

### **2.1. Merkmale als Tabelle**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_B/image_02_01.jpg?v=1784805292" width="250">



>* Signalfenster werden zu festen Tabellenzeilen
>* Merkmalsspalten machen scikit-learn-Training möglich

>* Jedes Fenster braucht den passenden Zielwert
>* Keine Zukunftsdaten in Merkmale einbauen

>* Wähle sinnvolle, stabile Signalmerkmale gezielt aus
>* Halte Tabellen konsistent für zuverlässige Pipelines



In [ ]:
#@title Python-Code - Merkmale als Tabelle

# Wir bauen Merkmale aus kurzen Signalfenstern.
# Die Tabelle speist eine scikit-learn-Pipeline.
# Zeitliche Trennung verhindert unrealistisch gute Ergebnisse.

import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Ein kleines künstliches Signal simuliert zwei Zustände.
rng = np.random.default_rng(42)
sample_rate_hz = 50
time_s = np.arange(0, 40, 1 / sample_rate_hz)

# Die zweite Hälfte enthält eine stärkere Schwingung.
base_wave = np.sin(2 * np.pi * 2 * time_s)
extra_wave = 0.8 * np.sin(2 * np.pi * 8 * time_s)
noise = rng.normal(0, 0.35, size=time_s.size)

signal = base_wave + noise
signal[time_s >= 20] = signal[time_s >= 20] + extra_wave[time_s >= 20]
labels = (time_s >= 20).astype(int)

# Jedes Fenster wird zu einer Tabellenzeile.
window_size = 100
step_size = 50
rows = []

for start in range(0, len(signal) - window_size + 1, step_size):
    end = start + window_size
    window = signal[start:end]
    target = int(labels[end - 1])

    spectrum = np.abs(np.fft.rfft(window))
    frequencies = np.fft.rfftfreq(window_size, d=1 / sample_rate_hz)
    dominant_frequency = frequencies[np.argmax(spectrum[1:]) + 1]

    rows.append(
        {
            "start_s": time_s[start],
            "mean": window.mean(),
            "std": window.std(),
            "energy": np.mean(window ** 2),
            "dominant_hz": dominant_frequency,
            "target": target,
        }
    )

feature_table = pd.DataFrame(rows)

# Eine einfache Prüfung macht die Tabellenform sichtbar.
expected_columns = {"mean", "std", "energy", "dominant_hz", "target"}
if not expected_columns.issubset(feature_table.columns):
    raise ValueError("Die Merkmalstabelle hat nicht alle erwarteten Spalten.")

# Zeitlich korrekt: frühe Fenster trainieren, spätere testen.
feature_names = ["mean", "std", "energy", "dominant_hz"]
split_index = int(len(feature_table) * 0.7)
train_table = feature_table.iloc[:split_index]
test_table = feature_table.iloc[split_index:]

X_train = train_table[feature_names]
y_train = train_table["target"]
X_test = test_table[feature_names]
y_test = test_table["target"]

# Die Pipeline skaliert nur mit Trainingsdaten.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=42, max_iter=200),
)
model.fit(X_train, y_train)

# Eine einfache Baseline sagt immer die häufigste Trainingsklasse voraus.
predictions = model.predict(X_test)
baseline_class = int(y_train.mode().iloc[0])
baseline_predictions = np.full_like(y_test.to_numpy(), baseline_class)

model_accuracy = accuracy_score(y_test, predictions)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
preview = feature_table[feature_names + ["target"]].head(3).round(2)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Merkmalstabelle: {feature_table.shape[0]} Fenster, {len(feature_names)} Merkmale")
print(preview.to_string(index=False))
print(f"Zeitlicher Test: Modell={model_accuracy:.2f}, Baseline={baseline_accuracy:.2f}")



### **2.2. Zeitlicher Split**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_B/image_02_02.jpg?v=1784805294" width="250">



>* Zeitreihen nicht zufällig mischen
>* Vergangenheit trainiert, Zukunft testet

>* Frühe Daten trainieren, spätere Daten testen
>* Vorverarbeitung erst nach zeitlicher Trennung

>* Passende zeitliche Einheit sorgfältig wählen
>* Mehrere Zeitabschnitte für stabile Bewertung nutzen



In [ ]:
#@title Python-Code - Zeitlicher Split

# Dieses Beispiel zeigt einen zeitlichen Split.
# Signalfenster bleiben in ihrer natürlichen Reihenfolge.
# Die Pipeline bewertet zukünftige Fenster realistisch.

import numpy as np
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir erzeugen ein kleines Signal mit zwei Zeitphasen.
rng = np.random.default_rng(42)
time_index = np.arange(240)

# Die spätere Phase hat höhere Schwingungsenergie.
base_signal = np.sin(time_index * 0.25)
noise = rng.normal(0.0, 0.25, size=time_index.size)

# Das Ziel beschreibt einen späteren Maschinenzustand.
signal = base_signal + noise + (time_index >= 120) * 0.9
target = (time_index >= 120).astype(int)

# Aus je zwölf Messpunkten entstehen einfache Fenstermerkmale.
window_size = 12
feature_rows = []
window_targets = []

for start in range(0, len(signal) - window_size + 1, window_size):
    window = signal[start:start + window_size]
    feature_rows.append([window.mean(), window.std()])
    window_targets.append(target[start + window_size - 1])

features = np.array(feature_rows)
labels = np.array(window_targets)

# Eine kurze Prüfung schützt vor unpassenden Fenstergrößen.
if len(features) < 10:
    raise ValueError("Zu wenige Fenster für dieses Beispiel.")

# Der Schnitt trennt Vergangenheit und Zukunft.
split_index = int(len(features) * 0.7)
X_train = features[:split_index]
y_train = labels[:split_index]

X_test = features[split_index:]
y_test = labels[split_index:]

# Skalierung wird nur auf Trainingsdaten gelernt.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=42, max_iter=200)
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)

# Eine einfache Baseline sagt immer die häufigste Trainingsklasse voraus.
majority_class = int(np.bincount(y_train).argmax())
baseline_predictions = np.full_like(y_test, majority_class)

model_accuracy = accuracy_score(y_test, predictions)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Trainingsfenster: {len(X_train)}, Testfenster: {len(X_test)}")
print(f"Zeitlicher Split: Modellgenauigkeit = {model_accuracy:.2f}")
print(f"Baseline: immer Klasse {majority_class} = {baseline_accuracy:.2f}")



### **2.3. Pipeline trainieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_B/image_02_03.jpg?v=1784805296" width="250">



>* Pipeline bündelt Vorverarbeitung und Modelltraining
>* Gleiche Schritte für neue Signalfenster

>* Nur Trainingsdaten zum Anpassen verwenden
>* So Datenleckagen und Zukunftswissen vermeiden

>* Lernverfahren passend zur Zielgröße wählen
>* Stabilität durch sauberes Training prüfen



In [ ]:
#@title Python-Code - Pipeline trainieren

# Wir trainieren eine Pipeline für Signalfenster.
# Der zeitliche Split verhindert Datenleckage.
# Die Ausgabe vergleicht Modell und Baseline.

import numpy as np
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Ein Zufallsgenerator macht die Signale reproduzierbar.
rng = np.random.default_rng(42)

# Wir erzeugen kleine Signalfenster in zeitlicher Reihenfolge.
sample_rate_hz = 50
window_size = 50
n_windows = 240

# Spätere Fenster haben häufiger die höhere Frequenzklasse.
time_axis = np.arange(window_size) / sample_rate_hz
labels = np.zeros(n_windows, dtype=int)
labels[120:] = 1

# Jedes Fenster enthält Sinusanteile und etwas Messrauschen.
signals = np.zeros((n_windows, window_size))
for index in range(n_windows):
    frequency_hz = 3 if labels[index] == 0 else 8
    noise = rng.normal(0, 0.35, size=window_size)
    signals[index] = np.sin(2 * np.pi * frequency_hz * time_axis) + noise

# Aus jedem Fenster berechnen wir einfache Signalmerkmale.
mean_feature = signals.mean(axis=1)
std_feature = signals.std(axis=1)
energy_feature = (signals ** 2).mean(axis=1)

# Die dominante Frequenz wird aus dem Spektrum geschätzt.
spectrum = np.abs(np.fft.rfft(signals, axis=1))
frequencies = np.fft.rfftfreq(window_size, d=1 / sample_rate_hz)
dominant_frequency = frequencies[np.argmax(spectrum[:, 1:], axis=1) + 1]

# Die Merkmale bilden die Eingabe für die Pipeline.
features = np.column_stack(
    [mean_feature, std_feature, energy_feature, dominant_frequency]
)

# Eine kurze Prüfung schützt vor stillen Formfehlern.
if features.shape != (n_windows, 4):
    raise ValueError("Die Merkmalsmatrix hat eine unerwartete Form.")

# Der Split folgt der Zeitachse, nicht einer zufälligen Mischung.
split_index = 160
X_train = features[:split_index]
y_train = labels[:split_index]
X_test = features[split_index:]
y_test = labels[split_index:]

# Skalierung und Modell werden gemeinsam nur auf Training angepasst.
pipeline = Pipeline(
    [("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=200))]
)
pipeline.fit(X_train, y_train)

# Eine einfache Baseline sagt immer die häufigste Trainingsklasse voraus.
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

# Beide Modelle werden nur auf späteren Testfenstern bewertet.
model_accuracy = accuracy_score(y_test, pipeline.predict(X_test))
baseline_accuracy = accuracy_score(y_test, baseline.predict(X_test))

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Trainingsfenster: {len(X_train)}, spätere Testfenster: {len(X_test)}")
print(f"Pipeline-Genauigkeit: {model_accuracy:.2f}")
print(f"Baseline-Genauigkeit: {baseline_accuracy:.2f}")



## **3. Signalmodelle zeitlich vergleichen**

### **3.1. Zeitliche Kreuzvalidierung**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_B/image_03_01.jpg?v=1784805298" width="250">



>* Nur Vergangenheit trainieren, Zukunft testen
>* Verhindert Datenleckage und zu optimistische Ergebnisse

>* Fortschreitend trainieren, auf spätere Abschnitte testen
>* Leistungsänderungen bei nichtstationären Signalen erkennen

>* Baselines zeitlich fair vergleichen
>* Robustheit über Zeitabschnitte prüfen



In [ ]:
#@title Python-Code - Zeitliche Kreuzvalidierung

# Wir vergleichen Signalmodelle mit zeitlicher Kreuzvalidierung.
# Zufällige Aufteilung kann bei Signalen täuschen.
# Die Ausgabe zeigt realistischere zukünftige Testfehler.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Ein kleines Signal verändert sein Verhalten langsam über die Zeit.
rng = np.random.default_rng(42)
time_index = np.arange(240)

trend = 0.02 * time_index
season = np.sin(time_index / 8)
noise = rng.normal(0, 0.25, size=time_index.size)
signal = trend + season + noise

# Aus vergangenen Messwerten entstehen einfache Fenstermerkmale.
window_size = 8
features = []
target = []

for end in range(window_size, len(signal)):
    window = signal[end - window_size:end]
    features.append([window.mean(), window.std(), window[-1]])
    target.append(signal[end])

X = np.array(features)
y = np.array(target)

# Diese Prüfung schützt vor unerwartet leeren Merkmalsmatrizen.
if X.shape[0] < 50 or X.shape[1] != 3:
    raise ValueError("Die erzeugten Fenster sind zu klein.")

# Eine Pipeline skaliert nur innerhalb des jeweiligen Trainingsabschnitts.
model = make_pipeline(StandardScaler(), LinearRegression())
baseline = DummyRegressor(strategy="mean")

# Zufällige Folds mischen Vergangenheit und Zukunft unrealistisch.
random_cv = KFold(n_splits=5, shuffle=True, random_state=42)
time_cv = TimeSeriesSplit(n_splits=5)

# Diese Funktion berechnet Fehler für mehrere zeitliche Prüfungen.
def collect_errors(splitter, estimator):
    errors = []
    for train_index, test_index in splitter.split(X):
        estimator.fit(X[train_index], y[train_index])
        prediction = estimator.predict(X[test_index])
        error = mean_absolute_error(y[test_index], prediction)
        errors.append(error)
    return np.array(errors)

random_errors = collect_errors(random_cv, model)
time_errors = collect_errors(time_cv, model)
baseline_errors = collect_errors(time_cv, baseline)

# Die kurzen Kennzahlen zeigen den Unterschied der Bewertungsarten.
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Zufällige Kreuzvalidierung, Modell-MAE: {random_errors.mean():.3f}")
print(f"Zeitliche Kreuzvalidierung, Modell-MAE: {time_errors.mean():.3f}")
print(f"Zeitliche Kreuzvalidierung, Baseline-MAE: {baseline_errors.mean():.3f}")

# Ein einzelnes Diagramm zeigt die Fehler je zukünftigem Testabschnitt.
fig, ax = plt.subplots(figsize=(8, 4))
fold_numbers = np.arange(1, len(time_errors) + 1)

ax.plot(fold_numbers, time_errors, marker="o", label="Modell")
ax.plot(fold_numbers, baseline_errors, marker="o", label="Baseline")
ax.set_title("Zeitliche Kreuzvalidierung für ein Signal")
ax.set_xlabel("Zeitlich späterer Testabschnitt")
ax.set_ylabel("Mittlerer absoluter Fehler")
ax.legend()
plt.show()



### **3.2. Vorhersagen sichtbar machen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_B/image_03_02.jpg?v=1784805300" width="250">



>* Vorhersagen als Zeitverlauf sichtbar machen
>* Fehlerphasen und Modellverhalten besser erkennen

>* Nur echte spätere Testabschnitte visualisieren
>* Zeitplots zeigen Verzögerungen und übersehene Ereignisse

>* Baselines zeigen den Nutzen komplexer Modelle
>* Zeitverläufe erklären wann Modelle besser sind



In [ ]:
#@title Python-Code - Vorhersagen sichtbar machen

# Wir machen Signalvorhersagen entlang der Zeit sichtbar.
# Der Testabschnitt bleibt zeitlich nach dem Training.
# Modell und Baseline werden direkt vergleichbar.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Ein kleines künstliches Signal simuliert Messwerte über Zeit.
rng = np.random.default_rng(42)
time = np.arange(240)

signal = 0.03 * time + np.sin(time / 8) + rng.normal(0, 0.25, size=time.size)
window_size = 8

# Aus jedem Fenster entstehen einfache Merkmale für den nächsten Wert.
features = []
targets = []

for end in range(window_size, len(signal)):
    window = signal[end - window_size:end]
    features.append([window[-1], window.mean(), window.std()])
    targets.append(signal[end])

features = np.array(features)
targets = np.array(targets)

# Die ersten Beobachtungen trainieren, spätere Beobachtungen testen.
split_index = 160
X_train = features[:split_index]
y_train = targets[:split_index]

X_test = features[split_index:]
y_test = targets[split_index:]
test_time = time[window_size + split_index:]

if len(X_test) == 0:
    raise ValueError("Der Testabschnitt darf nicht leer sein.")

# Die Baseline schreibt einfach den letzten Fensterwert fort.
baseline_pred = X_test[:, 0]

model = make_pipeline(StandardScaler(), LinearRegression())
model.fit(X_train, y_train)
model_pred = model.predict(X_test)

model_mae = mean_absolute_error(y_test, model_pred)
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"MAE Modell im späteren Testabschnitt: {model_mae:.3f}")
print(f"MAE Baseline im späteren Testabschnitt: {baseline_mae:.3f}")

# Eine Zeitgrafik zeigt, wann welches Verfahren danebenliegt.
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(test_time, y_test, label="Wahrer Wert", linewidth=2)
ax.plot(test_time, model_pred, label="Modellvorhersage", linewidth=2)
ax.plot(test_time, baseline_pred, label="Baseline: letzter Wert", linestyle="--")
ax.axvline(test_time[0], color="black", linestyle=":", label="Testbeginn")
ax.set_title("Vorhersagen im zeitlich späteren Testabschnitt")
ax.set_xlabel("Zeitpunkt")
ax.set_ylabel("Signalwert")
ax.legend()
plt.show()



### **3.3. Signal Mini Projekt**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_B/image_03_03.jpg?v=1784805302" width="250">



>* Nur Vergangenheit zum Trainieren nutzen
>* Spätere Signale realistisch testen

>* Klare Frage mit einfacher Baseline vergleichen
>* Vorhersagen zeitlich prüfen und Fehler erkennen

>* Begründe Modellwahl, Baseline und Grenzen.
>* Teste fair auf zukünftigen Signalabschnitten.



In [ ]:
#@title Python-Code - Signal Mini Projekt

# Dieses Mini-Projekt vergleicht Signalmodelle zeitlich fair.
# Zufälliges Mischen wirkt hier absichtlich zu optimistisch.
# Die Grafik zeigt Testfehler entlang der Zeit.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir erzeugen ein kleines Signal mit langsamem Drift.
rng = np.random.default_rng(42)
n_samples = 900
sample_index = np.arange(n_samples)

# Die Klassenregel ändert sich langsam über die Zeit.
phase = sample_index / 35.0
signal = np.sin(phase) + 0.35 * rng.normal(size=n_samples)
drift = 0.0015 * sample_index
labels = (signal + drift > 0.55).astype(int)

# Aus jedem Fenster entstehen einfache Signalmerkmale.
window_size = 30
feature_rows = []
target_values = []
window_times = []

for start in range(0, n_samples - window_size):
    window = signal[start:start + window_size]
    feature_rows.append([window.mean(), window.std(), window[-1] - window[0]])
    target_values.append(labels[start + window_size])
    window_times.append(start + window_size)

X = np.array(feature_rows)
y = np.array(target_values)
times = np.array(window_times)

# Eine kurze Prüfung macht die Datenannahme sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Der faire Schnitt trainiert nur auf früheren Fenstern.
split_point = int(0.7 * len(y))
X_train = X[:split_point]
y_train = y[:split_point]
X_test = X[split_point:]
y_test = y[split_point:]
test_times = times[split_point:]

# Das Modell nutzt Skalierung nur aus den Trainingsdaten.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=42, max_iter=300)
)
model.fit(X_train, y_train)

# Die Baseline sagt immer die häufigste Trainingsklasse voraus.
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

model_pred = model.predict(X_test)
baseline_pred = baseline.predict(X_test)
model_accuracy = accuracy_score(y_test, model_pred)
baseline_accuracy = accuracy_score(y_test, baseline_pred)

# Zum Vergleich zeigen wir auch eine unrealistische Zufallsteilung.
random_order = rng.permutation(len(y))
random_split = int(0.7 * len(y))
train_ids = random_order[:random_split]
test_ids = random_order[random_split:]

leaky_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=42, max_iter=300)
)
leaky_model.fit(X[train_ids], y[train_ids])
leaky_accuracy = accuracy_score(y[test_ids], leaky_model.predict(X[test_ids]))

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Zeitlicher Test: Modell={model_accuracy:.3f}, Baseline={baseline_accuracy:.3f}")
print(f"Zufällig gemischter Test: Modell={leaky_accuracy:.3f}")

# Die Fehlerkurven zeigen, wann Vorhersagen scheitern.
model_errors = (model_pred != y_test).astype(int)
baseline_errors = (baseline_pred != y_test).astype(int)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(test_times, baseline_errors, label="Baseline-Fehler", alpha=0.7)
ax.plot(test_times, model_errors, label="Modell-Fehler", alpha=0.7)
ax.set_title("Zeitlich korrekter Test auf späteren Signalfenstern")
ax.set_xlabel("Zeitindex des vorhergesagten Fensters")
ax.set_ylabel("Fehler: 1 ja, 0 nein")
ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Signalmodelle**</font>


In this lecture, you learned to:
- Erzeugen Fenster und Statistik- sowie Frequenzmerkmale aus kleinen Signalen. 
- Trainieren scikit-learn-Pipelines für Signal-Klassifikation oder Regression. 
- Bewerten Signalmodelle zeitlich korrekt und vergleichen sie mit Baselines. 

In the next Module (Module 14), we will go over 'Neuronale Netze'